# Part B methodology notebook v9: short-end traded maturities and wide-calibration comparison

This notebook consumes the richer Part A seed panel and keeps the existing hedging engine, while extending the
reporting to compare:

- **trade experiments**: baseline vs short-end
- **calibration windows**: local vs wide
- **maturity buckets**
- **regimes**

The point is not to rebuild the hedging engine again. It is to test *where* lifted Heston starts to matter more:
at the short end, under broader calibration windows, or both.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path("/mnt/data")

EPISODE_SEEDS_PATH = BASE / "episode_seeds_v15_short_wide.csv"
RESULTS_SEED_RAW_PATH = BASE / "part_b_v9_seed_results_raw.csv"
RESULTS_SEED_PATH = BASE / "part_b_v9_seed_results.csv"
RESULTS_SUMMARY_PATH = BASE / "part_b_v9_summary_results.csv"
RESULTS_BUCKET_PATH = BASE / "part_b_v9_bucket_summary_results.csv"
RESULTS_REGIME_PATH = BASE / "part_b_v9_regime_summary_results.csv"
RESULTS_STABILITY_PATH = BASE / "part_b_v9_training_stability_results.csv"
RESULTS_CI_PATH = BASE / "part_b_v9_bootstrap_ci.csv"

@dataclass
class PartBConfig:
    n_paths_train: int = 1024
    n_paths_val: int = 2048
    n_paths_test: int = 10_000
    min_steps: int = 5
    random_seed: int = 123

    run_worlds: tuple[str, ...] = ("heston_world", "lifted_world")
    benchmark_policies: tuple[str, ...] = (
        "unhedged", "bs_delta", "state_vol_delta", "fd_model_delta", "banded_fd_model_delta"
    )
    learned_policies: tuple[str, ...] = ("learned_hedge", "learned_hedge_tail")

    max_episode_seeds: int = 144
    train_fraction: float = 0.60
    val_fraction: float = 0.20

    tc_grid_bps: tuple[float, ...] = (0.25, 0.5, 1.0, 2.0)
    hedge_freq_grid: tuple[float, ...] = (0.5, 1.0, 2.0, 4.0)
    band_width: float = 0.10

    use_learned_hedge: bool = True
    learned_hidden: int = 32
    learned_epochs: int = 8
    learned_lr: float = 1e-3
    learned_risk_lambda: float = 0.25
    learned_tail_lambda: float = 0.50
    learned_tail_alpha: float = 0.05
    learned_train_runs: int = 5
    learned_contexts_per_epoch: int = 256

    fd_eps_rel: float = 1e-4
    continuation_ridge: float = 1e-6

    bootstrap_reps: int = 800
    bootstrap_seed: int = 2026

CFG_B = PartBConfig()
CFG_B


PartBConfig(n_paths_train=1024, n_paths_val=2048, n_paths_test=10000, min_steps=5, random_seed=123, run_worlds=('heston_world', 'lifted_world'), benchmark_policies=('unhedged', 'bs_delta', 'state_vol_delta', 'fd_model_delta', 'banded_fd_model_delta'), learned_policies=('learned_hedge', 'learned_hedge_tail'), max_episode_seeds=144, train_fraction=0.6, val_fraction=0.2, tc_grid_bps=(0.25, 0.5, 1.0, 2.0), hedge_freq_grid=(0.5, 1.0, 2.0, 4.0), band_width=0.1, use_learned_hedge=True, learned_hidden=32, learned_epochs=8, learned_lr=0.001, learned_risk_lambda=0.25, learned_tail_lambda=0.5, learned_tail_alpha=0.05, learned_train_runs=5, learned_contexts_per_epoch=256, fd_eps_rel=0.0001, continuation_ridge=1e-06, bootstrap_reps=800, bootstrap_seed=2026)

## Seed loading and experiment labels

The Part A seed file now contains explicit experiment metadata:

- `experiment_id`
- `trade_experiment`
- `calibration_window_mode`

The split is still chronological by trade date, but the reporting layer keeps these labels so we can ask whether:

- shorter traded maturities help lifted-Heston-based deep hedging
- wider calibration windows help lifted Heston on the hedging task


In [2]:
def assign_chronological_splits(seeds: pd.DataFrame, cfg: PartBConfig) -> pd.DataFrame:
    seeds = seeds.copy()
    seeds["trade_date"] = pd.to_datetime(seeds["trade_date"])
    unique_dates = np.sort(seeds["trade_date"].dropna().unique())
    n_dates = len(unique_dates)

    if n_dates <= 2:
        mapping = {d: ("train" if i == 0 else "test") for i, d in enumerate(unique_dates)}
    else:
        n_train = max(1, int(math.floor(n_dates * cfg.train_fraction)))
        n_val = max(1, int(math.floor(n_dates * cfg.val_fraction)))
        if n_train + n_val >= n_dates:
            n_val = max(1, n_dates - n_train - 1)
        train_dates = set(unique_dates[:n_train])
        val_dates = set(unique_dates[n_train:n_train + n_val])
        mapping = {}
        for d in unique_dates:
            if d in train_dates:
                mapping[d] = "train"
            elif d in val_dates:
                mapping[d] = "val"
            else:
                mapping[d] = "test"

    seeds["split"] = seeds["trade_date"].map(mapping).fillna("test")
    seeds["trade_date"] = seeds["trade_date"].dt.strftime("%Y-%m-%d")
    return seeds

def load_episode_seeds(cfg: PartBConfig) -> pd.DataFrame:
    if not EPISODE_SEEDS_PATH.exists():
        raise FileNotFoundError(f"Expected Part A seed file not found: {EPISODE_SEEDS_PATH}")

    seeds = pd.read_csv(EPISODE_SEEDS_PATH).copy()
    if "seed_id" not in seeds.columns:
        seeds["seed_id"] = [f"seed_{i:03d}" for i in range(len(seeds))]
    if "maturity_bucket" not in seeds.columns:
        seeds["maturity_bucket"] = np.where(seeds["dte"] < 11, "1w", np.where(seeds["dte"] < 21, "2w", np.where(seeds["dte"] < 45, "1m", "2m")))
    if "entry_iv" not in seeds.columns:
        seeds["entry_iv"] = np.sqrt(np.maximum(seeds.get("heston_v0", 0.04), 1e-10))
    if "trade_experiment" not in seeds.columns:
        seeds["trade_experiment"] = "baseline"
    if "calibration_window_mode" not in seeds.columns:
        seeds["calibration_window_mode"] = "local"
    if "experiment_id" not in seeds.columns:
        seeds["experiment_id"] = seeds["trade_experiment"].astype(str) + "_" + seeds["calibration_window_mode"].astype(str)

    if "regime_label" not in seeds.columns:
        seeds["trade_date"] = pd.to_datetime(seeds["trade_date"])
        date_iv = seeds.groupby("trade_date")["entry_iv"].median()
        threshold = float(date_iv.median()) if len(date_iv) else np.nan
        seeds["regime_iv_proxy"] = seeds["trade_date"].map(date_iv)
        seeds["regime_label"] = np.where(seeds["regime_iv_proxy"] >= threshold, "stressed", "calm")
        seeds["trade_date"] = seeds["trade_date"].dt.strftime("%Y-%m-%d")

    seeds = (
        seeds.sort_values(["trade_date", "experiment_id", "target_dte", "strike"])
        .head(cfg.max_episode_seeds)
        .reset_index(drop=True)
    )
    seeds = assign_chronological_splits(seeds, cfg)
    return seeds

episode_seeds = load_episode_seeds(CFG_B)
display(episode_seeds.head(12))
display(episode_seeds.groupby(["split", "experiment_id", "maturity_bucket"]).size().rename("n_seeds").reset_index().head(24))
display(episode_seeds.groupby(["split", "experiment_id", "regime_label"]).size().rename("n_seeds").reset_index().head(24))
print("n episode seeds:", len(episode_seeds))
print("trade dates:", episode_seeds["trade_date"].nunique())


,trade_date,experiment_id,trade_experiment,calibration_window_mode,spot_entry,r,q,heston_kappa,heston_theta,heston_sigma,...,put_ask_proxy,straddle_entry_mid,straddle_spread,call_rel_spread,put_rel_spread,pair_count_bucket,short_end_quality_flag,regime_iv_proxy,regime_label,split
0,2024-08-13,baseline_local,baseline,local,5434.430176,0.01,-0.029282,10.000000,0.027146,0.736527,...,66.000000,127.099998,1.399998,0.011391,0.010663,297,True,0.150229,calm,train
1,2024-08-13,baseline_local,baseline,local,5434.430176,0.01,-0.029282,10.000000,0.027146,0.736527,...,80.899994,191.899994,2.000000,0.009870,0.011187,482,False,0.150229,calm,train
2,2024-08-13,baseline_local,baseline,local,5434.430176,0.01,-0.029282,10.000000,0.027146,0.736527,...,132.899994,267.000000,1.599976,0.005948,0.006038,435,False,0.150229,calm,train
3,2024-08-13,baseline_wide,baseline,wide,5434.430176,0.01,-0.030132,8.943194,0.030993,0.767332,...,66.000000,127.099998,1.399998,0.011391,0.010663,296,True,0.150229,calm,train
4,2024-08-13,baseline_wide,baseline,wide,5434.430176,0.01,-0.030132,8.943194,0.030993,0.767332,...,80.899994,191.899994,2.000000,0.009870,0.011187,481,False,0.150229,calm,train
5,2024-08-13,baseline_wide,baseline,wide,5434.430176,0.01,-0.030132,8.943194,0.030993,0.767332,...,132.899994,267.000000,1.599976,0.005948,0.006038,430,False,0.150229,calm,train
6,2024-08-13,short_end_local,short_end,local,5434.430176,0.01,-0.024430,5.762230,0.005001,0.765951,...,49.500000,94.250000,1.099998,0.011099,0.012195,170,True,0.150229,calm,train
7,2024-08-13,short_end_local,short_end,local,5434.430176,0.01,-0.024430,5.762230,0.005001,0.765951,...,64.000000,128.000000,1.600002,0.012422,0.012579,295,True,0.150229,calm,train
8,2024-08-14,baseline_local,baseline,local,5455.209961,0.01,-0.039585,0.143653,0.500000,1.242728,...,54.799999,110.300003,1.199997,0.010753,0.011009,271,True,0.131327,calm,train
9,2024-08-14,baseline_local,baseline,local,5455.209961,0.01,-0.039585,0.143653,0.500000,1.242728,...,82.400009,166.700012,1.400002,0.008269,0.008531,428,False,0.131327,calm,train


,split,experiment_id,maturity_bucket,n_seeds
0,test,baseline_local,1m,5
1,test,baseline_local,2m,5
2,test,baseline_local,2w,5
3,test,baseline_wide,1m,5
4,test,baseline_wide,2m,5
5,test,baseline_wide,2w,5
6,test,short_end_local,1w,5
7,test,short_end_local,2w,5
8,train,baseline_local,1m,10
9,train,baseline_local,2m,10


,split,experiment_id,regime_label,n_seeds
0,test,baseline_local,calm,3
1,test,baseline_local,stressed,12
2,test,baseline_wide,calm,3
3,test,baseline_wide,stressed,12
4,test,short_end_local,calm,2
5,test,short_end_local,stressed,8
6,train,baseline_local,calm,18
7,train,baseline_local,stressed,12
8,train,baseline_wide,calm,18
9,train,baseline_wide,stressed,12


n episode seeds: 144
trade dates: 18


## Model worlds and continuation-value approximation

The important change is that each seed/world is simulated **once** on the finest hedge grid, then lower-frequency scenarios are evaluated by **subsampling** that base simulation. That is a cleaner use of compute than re-simulating each scenario from scratch.

In [3]:
def _fmt_num(x: float) -> str:
    return f"{float(x):g}".replace(".", "p")

def norm_cdf(x):
    x = np.asarray(x, dtype=float)
    return 0.5 * (1.0 + np.vectorize(math.erf)(x / math.sqrt(2.0)))

def bs_call_put_price_delta_vec(S, K, r, q, tau, sigma):
    S = np.asarray(S, dtype=float)
    sigma = np.maximum(np.asarray(sigma, dtype=float), 1e-10)
    tau = max(float(tau), 1e-10)
    K = float(K); r = float(r); q = float(q)
    if tau <= 1e-12:
        call = np.maximum(S - K, 0.0)
        put = np.maximum(K - S, 0.0)
        d_call = np.where(S > K, 1.0, np.where(S < K, 0.0, 0.5))
        d_put = d_call - 1.0
        return call, put, d_call, d_put
    vol = sigma * math.sqrt(tau)
    d1 = (np.log(np.maximum(S, 1e-12) / K) + (r - q + 0.5 * sigma * sigma) * tau) / vol
    d2 = d1 - vol
    Nd1 = norm_cdf(d1); Nd2 = norm_cdf(d2)
    Nmd1 = norm_cdf(-d1); Nmd2 = norm_cdf(-d2)
    disc_r = math.exp(-r * tau); disc_q = math.exp(-q * tau)
    call = disc_q * S * Nd1 - disc_r * K * Nd2
    put = disc_r * K * Nmd2 - disc_q * S * Nmd1
    d_call = disc_q * Nd1
    d_put = d_call - disc_q
    return call, put, d_call, d_put

def scenario_grid(cfg: PartBConfig) -> list[dict]:
    scenarios = []
    for tc_bps in cfg.tc_grid_bps:
        for hedge_steps_per_day in cfg.hedge_freq_grid:
            scenarios.append({
                "scenario_id": f"tc{_fmt_num(tc_bps)}_freq{_fmt_num(hedge_steps_per_day)}",
                "tc_bps": float(tc_bps),
                "hedge_steps_per_day": float(hedge_steps_per_day),
            })
    return scenarios

def n_paths_for_split(split: str, cfg: PartBConfig) -> int:
    return {"train": int(cfg.n_paths_train), "val": int(cfg.n_paths_val)}.get(str(split), int(cfg.n_paths_test))

def simulate_heston_paths(S0, r, q, kappa, theta, sigma, rho, v0, T, n_steps, n_paths, seed=123):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    sqrt_dt = math.sqrt(dt)
    S = np.empty((n_paths, n_steps + 1), dtype=np.float32)
    v = np.empty((n_paths, n_steps + 1), dtype=np.float32)
    S[:, 0] = S0
    v[:, 0] = max(v0, 1e-8)
    z1 = rng.standard_normal((n_paths, n_steps))
    z2 = rng.standard_normal((n_paths, n_steps))
    zs = rho * z1 + math.sqrt(max(1.0 - rho * rho, 1e-12)) * z2
    for t in range(n_steps):
        vt = np.maximum(v[:, t], 1e-10)
        v[:, t + 1] = np.maximum(v[:, t] + kappa * (theta - vt) * dt + sigma * np.sqrt(vt) * sqrt_dt * z1[:, t], 1e-10)
        S[:, t + 1] = S[:, t] * np.exp((r - q - 0.5 * vt) * dt + np.sqrt(vt) * sqrt_dt * zs[:, t])
    zeros = np.zeros_like(S, dtype=np.float32)
    return {"S": S, "v": v, "factor_fast": zeros, "factor_slow": zeros, "dt": float(dt)}

def lifted_factor_grid(n_factors):
    return np.geomspace(1.0, 64.0, int(n_factors)).astype(float)

def simulate_lifted_heston_paths(S0, r, q, H, kappa, theta, nu, rho, V0, T, n_steps, n_paths, n_factors=4, seed=123):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    sqrt_dt = math.sqrt(dt)
    lambdas = lifted_factor_grid(n_factors)
    weights = np.ones(int(n_factors), dtype=float)
    weights /= weights.sum()
    S = np.empty((n_paths, n_steps + 1), dtype=np.float32)
    v = np.empty((n_paths, n_steps + 1), dtype=np.float32)
    Y = np.zeros((n_paths, int(n_factors)), dtype=np.float32)
    factor_fast = np.empty((n_paths, n_steps + 1), dtype=np.float32)
    factor_slow = np.empty((n_paths, n_steps + 1), dtype=np.float32)
    S[:, 0] = S0
    v[:, 0] = max(V0, 1e-8)
    factor_fast[:, 0] = 0.0
    factor_slow[:, 0] = 0.0
    z1 = rng.standard_normal((n_paths, n_steps))
    z2 = rng.standard_normal((n_paths, n_steps))
    zv = z1
    zs = rho * z1 + math.sqrt(max(1.0 - rho * rho, 1e-12)) * z2
    rough_scale = max(dt ** max(H - 0.5, -0.49), 1.0)
    for t in range(n_steps):
        vt = np.maximum(v[:, t], 1e-10)
        common = np.sqrt(vt) * sqrt_dt * zv[:, t]
        Y = Y + kappa * (theta - vt)[:, None] * dt - lambdas[None, :] * Y * dt + nu * rough_scale * common[:, None]
        factor_fast[:, t + 1] = Y[:, 0]
        factor_slow[:, t + 1] = Y[:, -1]
        v[:, t + 1] = np.maximum(V0 + Y @ weights, 1e-10)
        S[:, t + 1] = S[:, t] * np.exp((r - q - 0.5 * vt) * dt + np.sqrt(vt) * sqrt_dt * zs[:, t])
    return {"S": S, "v": v, "factor_fast": factor_fast, "factor_slow": factor_slow, "dt": float(dt)}

def state_basis(S, K, v, factor_fast=None, factor_slow=None):
    S = np.asarray(S, dtype=float)
    x = np.log(np.maximum(S, 1e-12) / float(K))
    sv = np.sqrt(np.maximum(np.asarray(v, dtype=float), 1e-10))
    ff = np.zeros_like(x) if factor_fast is None else np.asarray(factor_fast, dtype=float)
    fs = np.zeros_like(x) if factor_slow is None else np.asarray(factor_slow, dtype=float)
    return np.column_stack([np.ones_like(x), x, x * x, sv, sv * x, ff, fs, x * ff, x * fs])

def fit_continuation_model(bundle: dict, seed_row: pd.Series, cfg: PartBConfig):
    S = bundle["S"]; v = bundle["v"]; ff = bundle["factor_fast"]; fs = bundle["factor_slow"]
    K = float(seed_row["strike"]); r = float(seed_row["r"]); T = float(seed_row["T"])
    payoff = np.maximum(S[:, -1] - K, 0.0) + np.maximum(K - S[:, -1], 0.0)
    betas = []
    for t in range(S.shape[1] - 1):
        tau = max(T - t * bundle["dt"], 1e-8)
        X = state_basis(S[:, t], K, v[:, t], ff[:, t], fs[:, t])
        y = math.exp(-r * tau) * payoff
        XtX = X.T @ X
        beta = np.linalg.solve(XtX + cfg.continuation_ridge * np.eye(X.shape[1]), X.T @ y)
        betas.append(beta.astype(np.float32))
    return {"betas": betas, "K": K}

def predict_fd_model_delta(continuation_model: dict, step: int, S, v, factor_fast, factor_slow, cfg: PartBConfig):
    beta = continuation_model["betas"][step]
    S = np.asarray(S, dtype=float)
    bump = np.maximum(np.abs(S) * cfg.fd_eps_rel, 1e-3)
    X_up = state_basis(S + bump, continuation_model["K"], v, factor_fast, factor_slow)
    X_dn = state_basis(np.maximum(S - bump, 1e-8), continuation_model["K"], v, factor_fast, factor_slow)
    return np.clip((X_up @ beta - X_dn @ beta) / (2.0 * bump), -1.5, 1.5).astype(np.float32)

def build_world_bundle(seed_row: pd.Series, world_name: str, cfg: PartBConfig) -> dict:
    max_freq = float(max(cfg.hedge_freq_grid))
    n_steps = max(int(round(float(seed_row["dte"]) * max_freq)), cfg.min_steps)
    n_paths = n_paths_for_split(seed_row["split"], cfg)
    seed_offset = abs(hash((seed_row["seed_id"], world_name, "base_bundle"))) % 100_000
    if world_name == "heston_world":
        bundle = simulate_heston_paths(
            S0=float(seed_row["spot_entry"]), r=float(seed_row["r"]), q=float(seed_row["q"]),
            kappa=float(seed_row["heston_kappa"]), theta=float(seed_row["heston_theta"]),
            sigma=float(seed_row["heston_sigma"]), rho=float(seed_row["heston_rho"]), v0=float(seed_row["heston_v0"]),
            T=float(seed_row["T"]), n_steps=n_steps, n_paths=n_paths, seed=cfg.random_seed + seed_offset,
        )
    else:
        bundle = simulate_lifted_heston_paths(
            S0=float(seed_row["spot_entry"]), r=float(seed_row["r"]), q=float(seed_row["q"]),
            H=float(seed_row["lifted_H"]), kappa=float(seed_row["lifted_kappa"]), theta=float(seed_row["lifted_theta"]),
            nu=float(seed_row["lifted_nu"]), rho=float(seed_row["lifted_rho"]), V0=float(seed_row["lifted_V0"]),
            T=float(seed_row["T"]), n_steps=n_steps, n_paths=n_paths,
            n_factors=int(seed_row.get("lifted_n_factors", 4)), seed=cfg.random_seed + 101 + seed_offset,
        )
    continuation = fit_continuation_model(bundle, seed_row, cfg)
    S = bundle["S"]; v = bundle["v"]; ff = bundle["factor_fast"]; fs = bundle["factor_slow"]
    fd_model_delta = np.empty((S.shape[0], S.shape[1]-1), dtype=np.float32)
    for t in range(S.shape[1]-1):
        fd_model_delta[:, t] = predict_fd_model_delta(continuation, t, S[:, t], v[:, t], ff[:, t], fs[:, t], cfg)
    payoff = np.maximum(S[:, -1] - float(seed_row["strike"]), 0.0) + np.maximum(float(seed_row["strike"]) - S[:, -1], 0.0)
    bundle["fd_model_delta"] = fd_model_delta
    bundle["fair_entry_model"] = float(math.exp(-float(seed_row["r"]) * float(seed_row["T"])) * payoff.mean())
    bundle["expected_unhedged_terminal_pnl"] = float(payoff.mean() - float(seed_row["straddle_entry_mid"]))
    return bundle

def make_scenario_view(bundle: dict, seed_row: pd.Series, scenario: dict, cfg: PartBConfig) -> dict:
    base_n_steps = bundle["S"].shape[1] - 1
    scenario_n_steps = max(int(round(float(seed_row["dte"]) * float(scenario["hedge_steps_per_day"]))), cfg.min_steps)
    step_idx = np.linspace(0, base_n_steps, scenario_n_steps + 1).round().astype(int)
    step_idx[0] = 0
    step_idx[-1] = base_n_steps
    step_idx = np.maximum.accumulate(step_idx)
    return {
        "S": bundle["S"][:, step_idx],
        "v": bundle["v"][:, step_idx],
        "factor_fast": bundle["factor_fast"][:, step_idx],
        "factor_slow": bundle["factor_slow"][:, step_idx],
        "fd_model_delta": bundle["fd_model_delta"][:, step_idx[:-1]],
        "time_index": step_idx,
        "base_dt": float(bundle["dt"]),
        "fair_entry_model": float(bundle["fair_entry_model"]),
        "expected_unhedged_terminal_pnl": float(bundle["expected_unhedged_terminal_pnl"]),
    }

def build_cached_worlds(seeds: pd.DataFrame, cfg: PartBConfig, cache_splits=("train", "val")) -> dict:
    cache = {}
    for _, seed_row in seeds.iterrows():
        if seed_row["split"] not in set(cache_splits):
            continue
        for world_name in cfg.run_worlds:
            cache[(seed_row["seed_id"], world_name)] = build_world_bundle(seed_row, world_name, cfg)
    return cache

scenarios = scenario_grid(CFG_B)
world_cache = build_cached_worlds(episode_seeds, CFG_B, cache_splits=("train", "val"))
print("n scenarios:", len(scenarios))
print("cached train/val bundles:", len(world_cache))

n scenarios: 16
cached train/val bundles: 208


## Hedge policies, 5-run stability, and the tail-aware extension

The learned hedge is trained **five times** per world. Instead of a separate network for every cost/frequency scenario, the policy gets transaction cost and hedge frequency as features. That keeps the training problem broad but tractable.

The extension here is a second learned policy, **`learned_hedge_tail`**, which adds a tail penalty aligned with the CVaR-style evaluation used later.

In [4]:
class HedgePolicy:
    name = "base"
    def target_batch(self, state: dict, context: dict) -> np.ndarray:
        raise NotImplementedError

class UnhedgedPolicy(HedgePolicy):
    name = "unhedged"
    def target_batch(self, state: dict, context: dict) -> np.ndarray:
        return np.zeros_like(state["S"], dtype=float)

class BSDeltaPolicy(HedgePolicy):
    name = "bs_delta"
    def target_batch(self, state: dict, context: dict) -> np.ndarray:
        _, _, d_call, d_put = bs_call_put_price_delta_vec(
            state["S"], state["K"], state["r"], state["q"], state["tau"], np.full_like(state["S"], state["entry_iv"])
        )
        return d_call + d_put

class StateVolDeltaPolicy(HedgePolicy):
    name = "state_vol_delta"
    def target_batch(self, state: dict, context: dict) -> np.ndarray:
        _, _, d_call, d_put = bs_call_put_price_delta_vec(
            state["S"], state["K"], state["r"], state["q"], state["tau"], np.sqrt(np.maximum(state["var_t"], 1e-10))
        )
        return d_call + d_put

class FDModelDeltaPolicy(HedgePolicy):
    name = "fd_model_delta"
    def target_batch(self, state: dict, context: dict) -> np.ndarray:
        return context["fd_model_delta"][:, context["step"]]

class BandedFDModelDeltaPolicy(HedgePolicy):
    name = "banded_fd_model_delta"
    def __init__(self, band: float):
        self.band = float(band)
    def target_batch(self, state: dict, context: dict) -> np.ndarray:
        raw = context["fd_model_delta"][:, context["step"]]
        current = state["inventory"]
        return np.where(np.abs(raw - current) <= self.band, current, raw)

BENCHMARK_POLICIES = {
    "unhedged": UnhedgedPolicy(),
    "bs_delta": BSDeltaPolicy(),
    "state_vol_delta": StateVolDeltaPolicy(),
    "fd_model_delta": FDModelDeltaPolicy(),
    "banded_fd_model_delta": BandedFDModelDeltaPolicy(CFG_B.band_width),
}

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

def build_feature_matrix_np(S, K, tau, r, q, var_t, inventory, factor_fast, factor_slow, model_delta, tc_bps, hedge_steps_per_day):
    S = np.asarray(S, dtype=float)
    return np.column_stack([
        S / float(K),
        np.log(np.maximum(S, 1e-12) / float(K)),
        np.full_like(S, float(tau)),
        np.full_like(S, float(r)),
        np.full_like(S, float(q)),
        np.sqrt(np.maximum(np.asarray(var_t, dtype=float), 1e-10)),
        np.asarray(inventory, dtype=float),
        np.asarray(factor_fast, dtype=float),
        np.asarray(factor_slow, dtype=float),
        np.asarray(model_delta, dtype=float),
        np.full_like(S, float(tc_bps) / 100.0),
        np.full_like(S, float(hedge_steps_per_day) / max(CFG_B.hedge_freq_grid)),
    ]).astype(np.float32)

if TORCH_AVAILABLE:
    class LearnedHedgeNet(nn.Module):
        def __init__(self, in_dim=12, hidden=32):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, 1),
            )
        def forward(self, x):
            return torch.tanh(self.net(x).squeeze(-1))

    class LearnedHedgePolicy(HedgePolicy):
        def __init__(self, name):
            self.name = str(name)
        def target_batch(self, state: dict, context: dict) -> np.ndarray:
            model = context.get("learned_model")
            if model is None:
                return context["fd_model_delta"][:, context["step"]]
            feats = build_feature_matrix_np(
                state["S"], state["K"], state["tau"], state["r"], state["q"], state["var_t"], state["inventory"],
                state["factor_fast"], state["factor_slow"], context["fd_model_delta"][:, context["step"]],
                context["tc_bps"], context["hedge_steps_per_day"],
            )
            with torch.no_grad():
                return model(torch.tensor(feats, dtype=torch.float32)).cpu().numpy()
else:
    class LearnedHedgePolicy(HedgePolicy):
        def __init__(self, name):
            self.name = str(name)
        def target_batch(self, state: dict, context: dict) -> np.ndarray:
            return context["fd_model_delta"][:, context["step"]]

LEARNED_POLICIES = {name: LearnedHedgePolicy(name) for name in CFG_B.learned_policies}

def differentiable_episode_loss(model, bundle: dict, seed_row: pd.Series, scenario: dict, policy_name: str, cfg: PartBConfig):
    view = make_scenario_view(bundle, seed_row, scenario, cfg)
    S = torch.tensor(view["S"], dtype=torch.float32)
    v = torch.tensor(view["v"], dtype=torch.float32)
    ff = torch.tensor(view["factor_fast"], dtype=torch.float32)
    fs = torch.tensor(view["factor_slow"], dtype=torch.float32)
    fd = torch.tensor(view["fd_model_delta"], dtype=torch.float32)
    inventory = torch.zeros(S.shape[0], dtype=torch.float32)
    cash = torch.full((S.shape[0],), -float(seed_row["straddle_entry_mid"]), dtype=torch.float32)
    tc = float(scenario["tc_bps"]) * 1e-4
    K = float(seed_row["strike"]); r = float(seed_row["r"]); q = float(seed_row["q"]); T = float(seed_row["T"])
    for t in range(S.shape[1] - 1):
        tau = max(T - view["time_index"][t] * view["base_dt"], 1e-8)
        feats = torch.tensor(
            build_feature_matrix_np(
                S[:, t].detach().cpu().numpy(), K, tau, r, q,
                v[:, t].detach().cpu().numpy(), inventory.detach().cpu().numpy(),
                ff[:, t].detach().cpu().numpy(), fs[:, t].detach().cpu().numpy(),
                fd[:, t].detach().cpu().numpy(), scenario["tc_bps"], scenario["hedge_steps_per_day"],
            ),
            dtype=torch.float32,
        )
        target = model(feats)
        trade = target - inventory
        cash = cash - trade * S[:, t] - tc * torch.abs(trade) * S[:, t]
        inventory = target
    payoff = torch.maximum(S[:, -1] - K, torch.zeros_like(S[:, -1])) + torch.maximum(K - S[:, -1], torch.zeros_like(S[:, -1]))
    pnl = cash + inventory * S[:, -1] + payoff
    loss = -pnl.mean() + CFG_B.learned_risk_lambda * pnl.std(unbiased=False)
    if policy_name == "learned_hedge_tail":
        k = max(1, int(math.ceil(cfg.learned_tail_alpha * S.shape[0])))
        tail_mean = torch.topk(pnl, k=k, largest=False).values.mean()
        loss = loss - cfg.learned_tail_lambda * tail_mean
    return loss

def train_learned_models(seeds: pd.DataFrame, cache: dict, scenarios: list[dict], cfg: PartBConfig):
    if (not TORCH_AVAILABLE) or (not cfg.use_learned_hedge):
        return {}
    models = {}
    rng = np.random.default_rng(cfg.random_seed)
    for world_name in cfg.run_worlds:
        train_items = [(seed_row, scenario) for _, seed_row in seeds.iterrows() if seed_row["split"] == "train" for scenario in scenarios]
        val_items = [(seed_row, scenario) for _, seed_row in seeds.iterrows() if seed_row["split"] == "val" for scenario in scenarios]
        for policy_name in cfg.learned_policies:
            for run_id in range(cfg.learned_train_runs):
                torch.manual_seed(cfg.random_seed + 1000 * run_id + (0 if world_name == "heston_world" else 100))
                model = LearnedHedgeNet(hidden=cfg.learned_hidden)
                optimizer = optim.Adam(model.parameters(), lr=cfg.learned_lr)
                best_state, best_val = None, np.inf
                for _ in range(cfg.learned_epochs):
                    model.train()
                    if len(train_items) > cfg.learned_contexts_per_epoch:
                        picks = rng.choice(len(train_items), size=cfg.learned_contexts_per_epoch, replace=False)
                        epoch_items = [train_items[i] for i in picks]
                    else:
                        epoch_items = train_items
                    for seed_row, scenario in epoch_items:
                        bundle = cache[(seed_row["seed_id"], world_name)]
                        loss = differentiable_episode_loss(model, bundle, seed_row, scenario, policy_name, cfg)
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                    val_losses = []
                    with torch.no_grad():
                        for seed_row, scenario in val_items[:min(len(val_items), cfg.learned_contexts_per_epoch)]:
                            bundle = cache[(seed_row["seed_id"], world_name)]
                            val_losses.append(float(differentiable_episode_loss(model, bundle, seed_row, scenario, policy_name, cfg)))
                    val_score = float(np.mean(val_losses)) if val_losses else 0.0
                    if val_score < best_val:
                        best_val = val_score
                        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                if best_state is not None:
                    model.load_state_dict(best_state)
                models[(world_name, policy_name, run_id)] = model.eval()
    return models

learned_models = train_learned_models(episode_seeds, world_cache, scenarios, CFG_B)
print("n learned models:", len(learned_models))

c:\Users\lhkke\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


n learned models: 20


## Evaluation, clustered bootstrap, and training-run summaries

The evaluation now works at the **seed-summary** level. That is the right granularity for:
- robust out-of-sample reporting,
- clustered bootstrap by **trade date**,
- and mean/std reporting across the **5 learned-hedge training runs**.

That is also how you avoid writing fifty-gigabyte CSV files just because you were feeling ambitious.

In [5]:
def get_world_bundle(seed_row: pd.Series, world_name: str, cache: dict, cfg: PartBConfig) -> dict:
    key = (seed_row["seed_id"], world_name)
    return cache[key] if key in cache else build_world_bundle(seed_row, world_name, cfg)

def simulate_policy_arrays(view: dict, seed_row: pd.Series, scenario: dict, policy: HedgePolicy, learned_model=None):
    S = view["S"]; v = view["v"]; ff = view["factor_fast"]; fs = view["factor_slow"]
    n_paths = S.shape[0]
    K = float(seed_row["strike"]); r = float(seed_row["r"]); q = float(seed_row["q"]); T = float(seed_row["T"])
    entry_iv = float(seed_row.get("entry_iv", math.sqrt(max(float(seed_row.get("heston_v0", 0.04)), 1e-10))))
    tc = float(scenario["tc_bps"]) * 1e-4
    inventory = np.zeros(n_paths, dtype=np.float32)
    cash = np.full(n_paths, -float(seed_row["straddle_entry_mid"]), dtype=np.float32)
    turnover = np.zeros(n_paths, dtype=np.float32)
    for t in range(S.shape[1] - 1):
        tau = max(T - view["time_index"][t] * view["base_dt"], 1e-8)
        state = {
            "S": S[:, t], "K": K, "tau": tau, "r": r, "q": q, "var_t": v[:, t],
            "inventory": inventory, "entry_iv": entry_iv, "factor_fast": ff[:, t], "factor_slow": fs[:, t],
        }
        pctx = {
            "step": t, "fd_model_delta": view["fd_model_delta"], "learned_model": learned_model,
            "tc_bps": scenario["tc_bps"], "hedge_steps_per_day": scenario["hedge_steps_per_day"],
        }
        target_inventory = policy.target_batch(state, pctx).astype(np.float32)
        trade = target_inventory - inventory
        cash -= trade * S[:, t] + tc * np.abs(trade) * S[:, t]
        inventory = target_inventory
        turnover += np.abs(trade)
    ST = S[:, -1]
    payoff = np.maximum(ST - K, 0.0) + np.maximum(K - ST, 0.0).astype(np.float32)
    final_pnl = cash + inventory * ST + payoff
    return np.asarray(final_pnl, dtype=float), np.asarray(turnover, dtype=float)

def summarize_policy_arrays(final_pnl, turnover, unhedged_pnl, seed_row: pd.Series, scenario: dict, world_name: str, policy_name: str, training_run, expected_unhedged_terminal_pnl: float):
    final_pnl = np.asarray(final_pnl, dtype=float)
    turnover = np.asarray(turnover, dtype=float)
    unhedged_pnl = np.asarray(unhedged_pnl, dtype=float)
    tail = final_pnl[final_pnl <= np.quantile(final_pnl, 0.05)]
    return {
        "scenario_id": scenario["scenario_id"],
        "tc_bps": float(scenario["tc_bps"]),
        "hedge_steps_per_day": float(scenario["hedge_steps_per_day"]),
        "seed_id": seed_row["seed_id"],
        "trade_date": seed_row["trade_date"],
        "split": seed_row["split"],
        "experiment_id": seed_row.get("experiment_id", "baseline_local"),
        "trade_experiment": seed_row.get("trade_experiment", "baseline"),
        "calibration_window_mode": seed_row.get("calibration_window_mode", "local"),
        "regime_label": seed_row.get("regime_label", "unknown"),
        "maturity_bucket": seed_row.get("maturity_bucket", "1m"),
        "world": world_name,
        "policy": policy_name,
        "training_run": int(training_run),
        "mean_pnl": float(np.mean(final_pnl)),
        "pnl_std": float(np.std(final_pnl, ddof=1)),
        "pnl_variance": float(np.var(final_pnl, ddof=1)),
        "cvar_5": float(np.mean(tail)) if len(tail) else np.nan,
        "mean_turnover": float(np.mean(turnover)),
        "mean_residual_pnl_vs_unhedged": float(np.mean(final_pnl - unhedged_pnl)),
        "mean_centered_pnl_vs_world_edge": float(np.mean(final_pnl - expected_unhedged_terminal_pnl)),
    }

def evaluate_seed_world(seed_row: pd.Series, world_name: str, cache: dict, scenarios: list[dict], learned_models: dict, cfg: PartBConfig):
    bundle = get_world_bundle(seed_row, world_name, cache, cfg)
    rows = []
    for scenario in scenarios:
        view = make_scenario_view(bundle, seed_row, scenario, cfg)
        base_arrays = {}
        for policy_name, policy in BENCHMARK_POLICIES.items():
            base_arrays[policy_name] = simulate_policy_arrays(view, seed_row, scenario, policy, learned_model=None)
        unhedged_pnl = base_arrays["unhedged"][0]
        expected_pnl = float(view["expected_unhedged_terminal_pnl"])
        for policy_name, (pnl, turnover) in base_arrays.items():
            rows.append(summarize_policy_arrays(pnl, turnover, unhedged_pnl, seed_row, scenario, world_name, policy_name, 0, expected_pnl))
        if CFG_B.use_learned_hedge and TORCH_AVAILABLE:
            for policy_name, policy in LEARNED_POLICIES.items():
                for run_id in range(CFG_B.learned_train_runs):
                    model = learned_models.get((world_name, policy_name, run_id))
                    pnl, turnover = simulate_policy_arrays(view, seed_row, scenario, policy, learned_model=model)
                    rows.append(summarize_policy_arrays(pnl, turnover, unhedged_pnl, seed_row, scenario, world_name, policy_name, run_id, expected_pnl))
    return rows

def collapse_seed_results(seed_results_raw: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["mean_pnl", "pnl_std", "pnl_variance", "cvar_5", "mean_turnover", "mean_residual_pnl_vs_unhedged", "mean_centered_pnl_vs_world_edge"]
    group_cols = [
        "scenario_id", "tc_bps", "hedge_steps_per_day", "split",
        "experiment_id", "trade_experiment", "calibration_window_mode",
        "regime_label", "world", "maturity_bucket", "seed_id", "trade_date", "policy"
    ]
    agg = seed_results_raw.groupby(group_cols)[metric_cols].agg(["mean", "std"])
    agg.columns = [f"{m}_{stat}" if stat == "std" else m for m, stat in agg.columns]
    seed_results = agg.reset_index()
    for c in [c for c in seed_results.columns if c.endswith("_std")]:
        seed_results[c] = seed_results[c].fillna(0.0)
    base = seed_results.loc[
        seed_results["policy"] == "unhedged",
        ["scenario_id", "seed_id", "world", "experiment_id", "pnl_variance"]
    ].rename(columns={"pnl_variance": "unhedged_variance"})
    seed_results = seed_results.merge(base, on=["scenario_id", "seed_id", "world", "experiment_id"], how="left")
    mask = seed_results["unhedged_variance"].notna() & (seed_results["unhedged_variance"] > 0)
    seed_results["seed_variance_reduction_vs_unhedged"] = np.where(mask, 1.0 - seed_results["pnl_variance"] / seed_results["unhedged_variance"], np.nan)
    return seed_results

def aggregate_panel(seed_results: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    metric_cols = ["mean_pnl", "pnl_std", "pnl_variance", "cvar_5", "mean_turnover", "mean_residual_pnl_vs_unhedged", "mean_centered_pnl_vs_world_edge", "seed_variance_reduction_vs_unhedged"]
    std_cols = [c for c in seed_results.columns if c.endswith("_std")]
    out = seed_results.groupby(group_cols).agg(**({"n_seeds": ("seed_id", "nunique")} | {c: (c, "mean") for c in metric_cols + std_cols})).reset_index()
    unhedged = out.loc[out["policy"] == "unhedged", group_cols + ["pnl_variance"]].rename(columns={"pnl_variance": "panel_unhedged_variance"})
    out = out.merge(unhedged, on=group_cols[:-1], how="left")
    mask = out["panel_unhedged_variance"].notna() & (out["panel_unhedged_variance"] > 0)
    out["hedging_error_variance"] = out["pnl_variance"]
    out["panel_variance_reduction_vs_unhedged"] = np.where(mask, 1.0 - out["pnl_variance"] / out["panel_unhedged_variance"], np.nan)
    return out

def build_training_stability(seed_results_raw: pd.DataFrame) -> pd.DataFrame:
    learned = seed_results_raw.loc[seed_results_raw["policy"].isin(CFG_B.learned_policies)].copy()
    if learned.empty:
        return pd.DataFrame()
    run_panel = learned.groupby([
        "scenario_id", "tc_bps", "hedge_steps_per_day", "split",
        "experiment_id", "trade_experiment", "calibration_window_mode",
        "world", "policy", "training_run"
    ]).agg(
        mean_pnl=("mean_pnl", "mean"),
        pnl_variance=("pnl_variance", "mean"),
        cvar_5=("cvar_5", "mean"),
        mean_turnover=("mean_turnover", "mean"),
        mean_residual_pnl_vs_unhedged=("mean_residual_pnl_vs_unhedged", "mean"),
    ).reset_index()
    stability = run_panel.groupby([
        "scenario_id", "tc_bps", "hedge_steps_per_day", "split",
        "experiment_id", "trade_experiment", "calibration_window_mode",
        "world", "policy"
    ]).agg(
        n_training_runs=("training_run", "nunique"),
        mean_pnl=("mean_pnl", "mean"),
        mean_pnl_run_std=("mean_pnl", "std"),
        pnl_variance=("pnl_variance", "mean"),
        pnl_variance_run_std=("pnl_variance", "std"),
        cvar_5=("cvar_5", "mean"),
        cvar_5_run_std=("cvar_5", "std"),
        mean_turnover=("mean_turnover", "mean"),
        mean_turnover_run_std=("mean_turnover", "std"),
        mean_residual_pnl_vs_unhedged=("mean_residual_pnl_vs_unhedged", "mean"),
        mean_residual_pnl_vs_unhedged_run_std=("mean_residual_pnl_vs_unhedged", "std"),
    ).reset_index()
    return stability.fillna(0.0)

def bootstrap_confidence_intervals(seed_results: pd.DataFrame, cfg: PartBConfig) -> pd.DataFrame:
    metrics = ["mean_residual_pnl_vs_unhedged", "pnl_variance", "mean_turnover", "cvar_5"]
    rows = []
    rng = np.random.default_rng(cfg.bootstrap_seed)
    group_keys = ["scenario_id", "split", "experiment_id", "trade_experiment", "calibration_window_mode", "world", "policy"]
    for group_key, grp in seed_results.groupby(group_keys):
        scenario_id, split, experiment_id, trade_experiment, calibration_window_mode, world, policy = group_key
        if split != "test":
            continue
        dates = np.array(sorted(pd.unique(grp["trade_date"])))
        if len(dates) < 2:
            continue
        date_map = {d: grp.loc[grp["trade_date"] == d] for d in dates}
        for metric in metrics:
            samples = []
            for _ in range(cfg.bootstrap_reps):
                draw = rng.choice(dates, size=len(dates), replace=True)
                boot = pd.concat([date_map[d] for d in draw], ignore_index=True)
                samples.append(float(boot[metric].mean()))
            lo, mid, hi = np.quantile(samples, [0.025, 0.50, 0.975])
            rows.append({
                "scenario_id": scenario_id,
                "split": split,
                "experiment_id": experiment_id,
                "trade_experiment": trade_experiment,
                "calibration_window_mode": calibration_window_mode,
                "world": world,
                "policy": policy,
                "metric": metric,
                "n_date_clusters": int(len(dates)),
                "estimate": float(grp[metric].mean()),
                "ci_lo": float(lo),
                "ci_mid": float(mid),
                "ci_hi": float(hi),
            })
    return pd.DataFrame(rows)

def evaluate_panel(seeds: pd.DataFrame, cache: dict, scenarios: list[dict], learned_models: dict, cfg: PartBConfig):
    raw_rows = []
    for _, seed_row in seeds.iterrows():
        for world_name in cfg.run_worlds:
            raw_rows.extend(evaluate_seed_world(seed_row, world_name, cache, scenarios, learned_models, cfg))
    seed_results_raw = pd.DataFrame(raw_rows)
    seed_results = collapse_seed_results(seed_results_raw)
    summary_results = aggregate_panel(
        seed_results,
        ["scenario_id", "tc_bps", "hedge_steps_per_day", "split", "experiment_id", "trade_experiment", "calibration_window_mode", "world", "policy"],
    )
    bucket_summary_results = aggregate_panel(
        seed_results,
        ["scenario_id", "tc_bps", "hedge_steps_per_day", "split", "experiment_id", "trade_experiment", "calibration_window_mode", "world", "maturity_bucket", "policy"],
    )
    regime_summary_results = aggregate_panel(
        seed_results,
        ["scenario_id", "tc_bps", "hedge_steps_per_day", "split", "experiment_id", "trade_experiment", "calibration_window_mode", "world", "regime_label", "policy"],
    )
    stability_results = build_training_stability(seed_results_raw)
    bootstrap_cis = bootstrap_confidence_intervals(seed_results, cfg)
    return seed_results_raw, seed_results, summary_results, bucket_summary_results, regime_summary_results, stability_results, bootstrap_cis


## Run the short-end / wide-calibration panel

The panel now keeps the original hedging engine, but compares experiments across:

- `baseline_local`
- `short_end_local`
- `baseline_wide`

The main question is whether lifted-Heston-based deep hedging improves more:

1. for **shorter traded maturities**, or
2. under **broader calibration windows**.


In [6]:
seed_results_raw, seed_results, summary_results, bucket_summary_results, regime_summary_results, stability_results, bootstrap_cis = evaluate_panel(
    episode_seeds, world_cache, scenarios, learned_models, CFG_B
)

seed_results_raw.to_csv(RESULTS_SEED_RAW_PATH, index=False)
seed_results.to_csv(RESULTS_SEED_PATH, index=False)
summary_results.to_csv(RESULTS_SUMMARY_PATH, index=False)
bucket_summary_results.to_csv(RESULTS_BUCKET_PATH, index=False)
regime_summary_results.to_csv(RESULTS_REGIME_PATH, index=False)
stability_results.to_csv(RESULTS_STABILITY_PATH, index=False)
bootstrap_cis.to_csv(RESULTS_CI_PATH, index=False)

display(summary_results.head(20))
display(bucket_summary_results.head(20))
display(regime_summary_results.head(20))
display(stability_results.head(20))
display(bootstrap_cis.head(20))
print("Saved:", RESULTS_SEED_RAW_PATH)
print("Saved:", RESULTS_SEED_PATH)
print("Saved:", RESULTS_SUMMARY_PATH)
print("Saved:", RESULTS_BUCKET_PATH)
print("Saved:", RESULTS_REGIME_PATH)
print("Saved:", RESULTS_STABILITY_PATH)
print("Saved:", RESULTS_CI_PATH)


,scenario_id,tc_bps,hedge_steps_per_day,split,experiment_id,trade_experiment,calibration_window_mode,world,policy_x,n_seeds,...,pnl_std_std,pnl_variance_std,cvar_5_std,mean_turnover_std,mean_residual_pnl_vs_unhedged_std,mean_centered_pnl_vs_world_edge_std,policy_y,panel_unhedged_variance,hedging_error_variance,panel_variance_reduction_vs_unhedged
0,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,banded_fd_model_delta,15,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,39435.613406,64873.469791,-0.645048
1,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,bs_delta,15,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,39435.613406,146354.718142,-2.711232
2,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,fd_model_delta,15,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,39435.613406,65461.499791,-0.659959
3,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,learned_hedge,15,...,5.568319,843.946136,4.617057,0.063163,1.229636,1.229636,unhedged,39435.613406,5387.877547,0.863375
4,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,learned_hedge_tail,15,...,3.521926,445.711977,3.774625,0.137255,1.123997,1.123997,unhedged,39435.613406,3849.534218,0.902384
5,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,state_vol_delta,15,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,39435.613406,129801.031860,-2.291467
6,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,unhedged,15,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,39435.613406,39435.613406,0.000000
7,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,lifted_world,banded_fd_model_delta,15,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,70437.813694,129580.978108,-0.839651
8,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,lifted_world,bs_delta,15,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,70437.813694,201651.423747,-1.862829
9,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,lifted_world,fd_model_delta,15,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,70437.813694,129867.030003,-0.843712


,scenario_id,tc_bps,hedge_steps_per_day,split,experiment_id,trade_experiment,calibration_window_mode,world,maturity_bucket,policy_x,...,pnl_std_std,pnl_variance_std,cvar_5_std,mean_turnover_std,mean_residual_pnl_vs_unhedged_std,mean_centered_pnl_vs_world_edge_std,policy_y,panel_unhedged_variance,hedging_error_variance,panel_variance_reduction_vs_unhedged
0,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,1m,banded_fd_model_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,33675.116317,50319.882482,-0.494275
1,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,1m,bs_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,33675.116317,124467.320363,-2.696121
2,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,1m,fd_model_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,33675.116317,50829.701292,-0.509414
3,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,1m,learned_hedge,...,5.584375,777.243700,4.037938,0.043881,1.323174,1.323174,unhedged,33675.116317,4558.385198,0.864636
4,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,1m,learned_hedge_tail,...,3.873219,456.624944,3.784031,0.120217,1.204090,1.204090,unhedged,33675.116317,3452.943143,0.897463
5,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,1m,state_vol_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,33675.116317,111811.526839,-2.320301
6,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,1m,unhedged,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,33675.116317,33675.116317,0.000000
7,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,2m,banded_fd_model_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,70249.961601,122245.725220,-0.740154
8,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,2m,bs_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,70249.961601,266544.968371,-2.794236
9,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,2m,fd_model_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,70249.961601,123479.362381,-0.757714


,scenario_id,tc_bps,hedge_steps_per_day,split,experiment_id,trade_experiment,calibration_window_mode,world,regime_label,policy_x,...,pnl_std_std,pnl_variance_std,cvar_5_std,mean_turnover_std,mean_residual_pnl_vs_unhedged_std,mean_centered_pnl_vs_world_edge_std,policy_y,panel_unhedged_variance,hedging_error_variance,panel_variance_reduction_vs_unhedged
0,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,calm,banded_fd_model_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,24285.976098,56174.108952,-1.313027
1,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,calm,bs_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,24285.976098,92597.373983,-2.812792
2,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,calm,fd_model_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,24285.976098,56263.481780,-1.316707
3,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,calm,learned_hedge,...,9.121343,1478.511810,5.542592,0.056710,1.510033,1.510033,unhedged,24285.976098,6454.262537,0.734239
4,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,calm,learned_hedge_tail,...,4.686031,553.395800,3.004979,0.113856,1.327101,1.327101,unhedged,24285.976098,3411.942310,0.859510
5,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,calm,state_vol_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,24285.976098,87810.377982,-2.615682
6,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,calm,unhedged,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,24285.976098,24285.976098,0.000000
7,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,stressed,banded_fd_model_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,43223.022733,67048.310001,-0.551218
8,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,stressed,bs_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,43223.022733,159794.054182,-2.696966
9,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,stressed,fd_model_delta,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unhedged,43223.022733,67761.004294,-0.567706


,scenario_id,tc_bps,hedge_steps_per_day,split,experiment_id,trade_experiment,calibration_window_mode,world,policy,n_training_runs,mean_pnl,mean_pnl_run_std,pnl_variance,pnl_variance_run_std,cvar_5,cvar_5_run_std,mean_turnover,mean_turnover_run_std,mean_residual_pnl_vs_unhedged,mean_residual_pnl_vs_unhedged_run_std
0,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,learned_hedge,5,1.044322,1.195820,5387.877547,415.705748,-135.151483,2.167252,1.494967,0.037521,5.553422,1.195820
1,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,learned_hedge_tail,5,-2.693005,1.102695,3849.534218,186.777113,-108.895142,2.536028,2.042723,0.112911,1.816095,1.102695
2,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,lifted_world,learned_hedge,5,-104.450175,0.070143,29538.635920,776.945424,-207.978714,0.971674,1.191567,0.030553,21.387823,0.070143
3,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,lifted_world,learned_hedge_tail,5,-105.838889,0.817426,29641.979536,1108.156348,-203.962928,1.633809,1.205574,0.069831,19.999109,0.817426
4,tc0p25_freq0p5,0.25,0.5,test,baseline_wide,baseline,wide,heston_world,learned_hedge,5,6.131110,1.182758,10787.724408,706.681345,-149.867573,1.120116,1.593133,0.061212,5.031285,1.182758
5,tc0p25_freq0p5,0.25,0.5,test,baseline_wide,baseline,wide,heston_world,learned_hedge_tail,5,2.747973,1.043540,9552.035518,214.532312,-131.797318,1.469200,2.038478,0.105183,1.648148,1.043540
6,tc0p25_freq0p5,0.25,0.5,test,baseline_wide,baseline,wide,lifted_world,learned_hedge,5,-91.835186,0.299260,46392.187658,15099.208811,-210.437502,1.585622,1.198864,0.035135,21.956133,0.299260
7,tc0p25_freq0p5,0.25,0.5,test,baseline_wide,baseline,wide,lifted_world,learned_hedge_tail,5,-93.221635,0.793001,51098.858092,12237.743589,-203.314326,2.201613,1.222058,0.098455,20.569684,0.793001
8,tc0p25_freq0p5,0.25,0.5,test,short_end_local,short_end,local,heston_world,learned_hedge,5,2.106959,0.310995,8490.558281,335.842982,-127.184317,0.936601,0.837222,0.058122,0.952014,0.310995
9,tc0p25_freq0p5,0.25,0.5,test,short_end_local,short_end,local,heston_world,learned_hedge_tail,5,1.743600,0.277831,8060.603207,433.464565,-121.636850,1.444423,1.006682,0.086707,0.588656,0.277831


,scenario_id,split,experiment_id,trade_experiment,calibration_window_mode,world,policy,metric,n_date_clusters,estimate,ci_lo,ci_mid,ci_hi
0,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,banded_fd_model_delta,mean_residual_pnl_vs_unhedged,5,2.329796,1.636872,2.306291,3.448673
1,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,banded_fd_model_delta,pnl_variance,5,64873.469791,58381.517313,64873.469791,72687.707463
2,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,banded_fd_model_delta,mean_turnover,5,1.933345,1.539668,1.933345,2.327023
3,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,banded_fd_model_delta,cvar_5,5,-309.016210,-337.107510,-309.016210,-280.851028
4,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,bs_delta,mean_residual_pnl_vs_unhedged,5,1.625264,0.146028,1.625264,3.035212
5,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,bs_delta,pnl_variance,5,146354.718142,117463.639032,147900.726922,168385.848120
6,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,bs_delta,mean_turnover,5,2.348732,2.286271,2.347952,2.439723
7,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,bs_delta,cvar_5,5,-453.775363,-484.920867,-454.756998,-412.150555
8,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,fd_model_delta,mean_residual_pnl_vs_unhedged,5,2.233412,1.529355,2.180655,3.363553
9,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,fd_model_delta,pnl_variance,5,65461.499791,59251.176131,65461.499791,73269.303877


Saved: \mnt\data\part_b_v9_seed_results_raw.csv
Saved: \mnt\data\part_b_v9_seed_results.csv
Saved: \mnt\data\part_b_v9_summary_results.csv
Saved: \mnt\data\part_b_v9_bucket_summary_results.csv
Saved: \mnt\data\part_b_v9_regime_summary_results.csv
Saved: \mnt\data\part_b_v9_training_stability_results.csv
Saved: \mnt\data\part_b_v9_bootstrap_ci.csv


## Headline plots and tables by experiment

The headline view now compares the learned policies and the unhedged benchmark across
the experiment labels rather than trying to cram every benchmark policy into one plot.


In [10]:
def _ensure_panel_frame(df: pd.DataFrame, fallback: pd.DataFrame | None = None) -> pd.DataFrame:
    if isinstance(df, pd.DataFrame) and "policy" in df.columns:
        return df
    if isinstance(df, pd.DataFrame):
        if isinstance(df.index, pd.MultiIndex) and "policy" in list(df.index.names):
            return df.reset_index()
        if getattr(df.index, "name", None) == "policy":
            return df.reset_index()
    if fallback is not None and isinstance(fallback, pd.DataFrame) and "policy" in fallback.columns:
        return fallback
    return pd.DataFrame()

summary_results_plot = _ensure_panel_frame(
    summary_results,
    fallback=aggregate_panel(
        seed_results,
        ["scenario_id", "tc_bps", "hedge_steps_per_day", "split", "experiment_id", "trade_experiment", "calibration_window_mode", "world", "policy"],
    ) if "seed_results" in globals() and isinstance(seed_results, pd.DataFrame) and not seed_results.empty else None,
)
bucket_summary_plot = _ensure_panel_frame(
    bucket_summary_results,
    fallback=aggregate_panel(
        seed_results,
        ["scenario_id", "tc_bps", "hedge_steps_per_day", "split", "experiment_id", "trade_experiment", "calibration_window_mode", "world", "maturity_bucket", "policy"],
    ) if "seed_results" in globals() and isinstance(seed_results, pd.DataFrame) and not seed_results.empty else None,
)
regime_summary_plot = _ensure_panel_frame(
    regime_summary_results,
    fallback=aggregate_panel(
        seed_results,
        ["scenario_id", "tc_bps", "hedge_steps_per_day", "split", "experiment_id", "trade_experiment", "calibration_window_mode", "world", "regime_label", "policy"],
    ) if "seed_results" in globals() and isinstance(seed_results, pd.DataFrame) and not seed_results.empty else None,
)

headline_policies = ["unhedged", "learned_hedge", "learned_hedge_tail"]

if not summary_results_plot.empty:
    print("sumnmary_results_plot not empty")
    headline_scenario = scenarios[0]["scenario_id"]
    plot_df = summary_results_plot.loc[
        (summary_results_plot["split"] == "test")
        & (summary_results_plot["scenario_id"] == headline_scenario)
        & (summary_results_plot["policy"].isin(headline_policies))
    ].copy()
    plot_df["experiment_world"] = plot_df["experiment_id"].astype(str) + " | " + plot_df["world"].astype(str)

    if not plot_df.empty and {"experiment_world", "policy"}.issubset(plot_df.columns):
        fig, axes = plt.subplots(2, 2, figsize=(18, 10))
        for ax, metric, title in [
            (axes[0, 0], "mean_residual_pnl_vs_unhedged", "Test residual P&L vs unhedged"),
            (axes[0, 1], "pnl_variance", "Test raw P&L variance"),
            (axes[1, 0], "mean_turnover", "Test mean turnover"),
            (axes[1, 1], "cvar_5", "Test CVaR 5%"),
        ]:
            plot_df.pivot(index="experiment_world", columns="policy", values=metric).plot(kind="bar", ax=ax, title=title)
            ax.set_ylabel(metric)
        fig.tight_layout()
        display(fig)
        plt.close(fig)
    else:
        print("Plotting skipped: summary frame does not contain expected experiment/policy columns.")
        print("Available columns:", list(plot_df.columns))




In [8]:
headline_cols = [
    "scenario_id", "split", "experiment_id", "trade_experiment", "calibration_window_mode", "world", "policy",
    "n_seeds", "mean_residual_pnl_vs_unhedged", "pnl_variance", "mean_turnover", "cvar_5", "panel_variance_reduction_vs_unhedged"
]
if not summary_results_plot.empty:
    display(
        summary_results_plot.loc[
            (summary_results_plot["split"] == "test") & summary_results_plot["policy"].isin(headline_policies),
            headline_cols,
        ].sort_values(["scenario_id", "experiment_id", "world", "policy"]).head(60)
    )
if not bucket_summary_plot.empty:
    display(
        bucket_summary_plot.loc[
            (bucket_summary_plot["split"] == "test") & bucket_summary_plot["policy"].isin(headline_policies),
            ["scenario_id", "experiment_id", "world", "maturity_bucket", "policy", "n_seeds", "mean_residual_pnl_vs_unhedged", "pnl_variance", "mean_turnover", "cvar_5"],
        ].sort_values(["scenario_id", "experiment_id", "world", "maturity_bucket", "policy"]).head(60)
    )
if not regime_summary_plot.empty:
    display(
        regime_summary_plot.loc[
            (regime_summary_plot["split"] == "test") & regime_summary_plot["policy"].isin(headline_policies),
            ["scenario_id", "experiment_id", "world", "regime_label", "policy", "n_seeds", "mean_residual_pnl_vs_unhedged", "pnl_variance", "mean_turnover", "cvar_5"],
        ].sort_values(["scenario_id", "experiment_id", "world", "regime_label", "policy"]).head(60)
    )
if "stability_results" in globals() and isinstance(stability_results, pd.DataFrame) and not stability_results.empty:
    display(
        stability_results.loc[
            (stability_results["split"] == "test") & stability_results["policy"].isin(["learned_hedge", "learned_hedge_tail"])
        ].sort_values(["scenario_id", "experiment_id", "world", "policy"]).head(60)
    )
if "bootstrap_cis" in globals() and isinstance(bootstrap_cis, pd.DataFrame) and not bootstrap_cis.empty:
    display(
        bootstrap_cis.loc[bootstrap_cis["policy"].isin(["learned_hedge", "learned_hedge_tail"])]
        .sort_values(["scenario_id", "experiment_id", "world", "policy", "metric"]).head(80)
    )

,scenario_id,tc_bps,hedge_steps_per_day,split,experiment_id,trade_experiment,calibration_window_mode,world,policy,n_training_runs,mean_pnl,mean_pnl_run_std,pnl_variance,pnl_variance_run_std,cvar_5,cvar_5_run_std,mean_turnover,mean_turnover_run_std,mean_residual_pnl_vs_unhedged,mean_residual_pnl_vs_unhedged_run_std
0,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,learned_hedge,5,1.044322,1.195820,5387.877547,415.705748,-135.151483,2.167252,1.494967,0.037521,5.553422,1.195820
1,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,heston_world,learned_hedge_tail,5,-2.693005,1.102695,3849.534218,186.777113,-108.895142,2.536028,2.042723,0.112911,1.816095,1.102695
2,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,lifted_world,learned_hedge,5,-104.450175,0.070143,29538.635920,776.945424,-207.978714,0.971674,1.191567,0.030553,21.387823,0.070143
3,tc0p25_freq0p5,0.25,0.5,test,baseline_local,baseline,local,lifted_world,learned_hedge_tail,5,-105.838889,0.817426,29641.979536,1108.156348,-203.962928,1.633809,1.205574,0.069831,19.999109,0.817426
4,tc0p25_freq0p5,0.25,0.5,test,baseline_wide,baseline,wide,heston_world,learned_hedge,5,6.131110,1.182758,10787.724408,706.681345,-149.867573,1.120116,1.593133,0.061212,5.031285,1.182758
5,tc0p25_freq0p5,0.25,0.5,test,baseline_wide,baseline,wide,heston_world,learned_hedge_tail,5,2.747973,1.043540,9552.035518,214.532312,-131.797318,1.469200,2.038478,0.105183,1.648148,1.043540
6,tc0p25_freq0p5,0.25,0.5,test,baseline_wide,baseline,wide,lifted_world,learned_hedge,5,-91.835186,0.299260,46392.187658,15099.208811,-210.437502,1.585622,1.198864,0.035135,21.956133,0.299260
7,tc0p25_freq0p5,0.25,0.5,test,baseline_wide,baseline,wide,lifted_world,learned_hedge_tail,5,-93.221635,0.793001,51098.858092,12237.743589,-203.314326,2.201613,1.222058,0.098455,20.569684,0.793001
8,tc0p25_freq0p5,0.25,0.5,test,short_end_local,short_end,local,heston_world,learned_hedge,5,2.106959,0.310995,8490.558281,335.842982,-127.184317,0.936601,0.837222,0.058122,0.952014,0.310995
9,tc0p25_freq0p5,0.25,0.5,test,short_end_local,short_end,local,heston_world,learned_hedge_tail,5,1.743600,0.277831,8060.603207,433.464565,-121.636850,1.444423,1.006682,0.086707,0.588656,0.277831


,scenario_id,split,experiment_id,trade_experiment,calibration_window_mode,world,policy,metric,n_date_clusters,estimate,ci_lo,ci_mid,ci_hi
15,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,learned_hedge,cvar_5,5,-135.151483,-142.133730,-135.253565,-124.268575
12,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,learned_hedge,mean_residual_pnl_vs_unhedged,5,5.553422,3.630556,5.574122,7.556768
14,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,learned_hedge,mean_turnover,5,1.494967,1.456557,1.494967,1.532669
13,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,learned_hedge,pnl_variance,5,5387.877547,4986.491395,5373.275971,5951.364576
19,tc0p25_freq0p5,test,baseline_local,baseline,local,heston_world,learned_hedge_tail,cvar_5,5,-108.895142,-115.482618,-108.994405,-99.695256
...,...,...,...,...,...,...,...,...,...,...,...,...,...
265,tc0p25_freq1,test,baseline_wide,baseline,wide,lifted_world,learned_hedge,pnl_variance,5,44449.982313,6862.901512,41948.187355,107062.594422
271,tc0p25_freq1,test,baseline_wide,baseline,wide,lifted_world,learned_hedge_tail,cvar_5,5,-200.583586,-219.466684,-200.583586,-174.956410
268,tc0p25_freq1,test,baseline_wide,baseline,wide,lifted_world,learned_hedge_tail,mean_residual_pnl_vs_unhedged,5,20.488904,17.775963,20.606859,22.809354
270,tc0p25_freq1,test,baseline_wide,baseline,wide,lifted_world,learned_hedge_tail,mean_turnover,5,1.443933,1.369981,1.443933,1.511794


## Where this notebook now stands

This version keeps the stronger evaluation design, but uses the richer Part A panel to answer two additional questions:

1. **Short-end trade test**: does lifted-Heston-based learned hedging add more value for `1w` / `2w` traded straddles?
2. **Wide-calibration test**: does lifted Heston improve when the calibration problem is made more multi-timescale?

The intent is to preserve the baseline 30d-style benchmark while creating a cleaner experimental path toward settings in which lifted Heston should plausibly matter more.
